# SuperEEG leave-one-out analysis

The objective is to measure the quality of the SuperEEG model. We perform a leave-one-out analysis and evaluate the quality of the reconstruction. We can measure the quality of the model by examining statistics such as average and histogrammed RMSE and metrics of individual traces.

The leave-one-out analysis looks like this:

1. load in all the data (resample to 250Hz ?)
2. cull electrodes which do not pass a kurtosis test
3. cull brains with fewer than two (three?) remaining electrodes
4. compile all remaining electrode locations
5. for each brain:
    1. compute full-brain collelation matrix K using all other patients' data (build models from brains using locs, then build one model from those models)
    2. for each electrode in this brain:
        1. extract Y_ska by removing this electrode
        2. compute Y_skb = (K_ba*inv(K_aa) * Y_ska.T).T
        3. compare Y_skb with observed trace
6. do something with the predicted traces. correlation

### Load in the data

let's start off with a small dataset - DANDI 000576. after loading them, compile the electrode locations and make brain objects with those locations

In [1]:
import supereeg as se
import os
import numpy as np
from scipy.io import loadmat
from pynwb import NWBHDF5IO
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import multiprocessing as mp
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [2]:
# needs updated to handle multiple sessions
def load_Miller():
    task = 'faceshouses-basic'
    data_dir = f'../brain-lab-data/miller-BIDS/{task}'
    data_name = f'{task}_ieeg.mat'
    locs_name = 'electrodes.mat'
    
    # read the subject ids from the directory
    IDS = [x[-2:] for x in os.listdir(data_dir) if x.startswith('sub-') and 'jm' not in x]
    
    # read in the data
    DATA = {f'{x}': loadmat(f"{data_dir}/sub-{x}/ieeg/sub-{x}_{data_name}") for x in IDS}
    
    # read in the electrode locations
    for x in IDS:
        DATA[x]['locs'] = loadmat(f"{data_dir}/sub-{x}/ieeg/sub-{x}_{locs_name}")['locs']
    
    # compile all locations
    R = np.unique(np.concatenate([DATA[x]['locs'] for x in IDS]),axis=0)
    
    # make brain objects for each subject
    BOS = {f'{x}': se.Brain(data = DATA[x]['data'], locs = DATA[x]['locs'], 
                    sample_rate = 1000) for x in IDS}

    return IDS, DATA, R, BOS

In [122]:
def load_DANDI(data_dir, read_cmd):
    
    # read the subject ids from the directory
    IDS = [x.split("-")[1] for x in os.listdir(data_dir) if x.startswith('sub-')]

    # read in the data
    DATA = {}
    pop_list = []
    for x in IDS:
        
        # make data structure
        DATA[x] = {}
        DATA[x]['sessions'] = []

        # iterate over sessions
        for file in os.listdir(sorted(Path(f'{data_dir}').glob(f'sub-{x}*'))[0]):
            if file.endswith('.nwb'):
                
                # get the session name
                sess = file.split("_")[1].split("-")[1]
                if sess in DATA[x]['sessions']: continue
                
                # open the file with HDPy
                f = NWBHDF5IO(sorted(Path(f'{data_dir}').glob(f'sub-{x}*/*{sess}*.nwb'))[0], "r").read()
                
                # read from file
                try:
                    # get electrode locations
                    # ASSUMING ALL SESSIONS HAVE SAME ELECTRODES
                    trodes = f.electrodes[:][['x','y','z']].to_numpy()
                    if np.any(np.isnan(trodes)): raise Exception()
                    DATA[x]['locs'] = trodes
                    print(f'session {f.identifier} has locs')
                    
                    # get electrode readings
                    try:
                        DATA[x]['data'] = np.vstack((DATA[x]['data'], eval(read_cmd)))
                    except:
                        DATA[x]['data'] = eval(read_cmd).astype(float)
                        print(DATA[x]['data'].shape)
                        N = DATA[x]['data'].shape[0]
                    
                    # save session name
                    if len(DATA[x]['sessions']):
                        DATA[x]['sessions'] += [sess]*N
                    else:
                        DATA[x]['sessions'] = [sess]*N
                
                except:
                    print(f'locs not available for {f.identifier}')
                    pop_list.append(x)
    
    # remove IDS of datasets with no locations
    for item in pop_list:
        if item in IDS: IDS.remove(item)
    
    # compile all locations
    R = np.unique(np.vstack([DATA[x]['locs'] for x in IDS]))
    
    # make brain objects for each subject
    # ASSUMING ALL SESSIONS ARE THE SAME LENGTH
    BOS = {f'{x}': se.Brain(data = DATA[x]['data'], locs = DATA[x]['locs'], 
            sessions = DATA[x]['sessions'], sample_rate = 2000) for x in IDS}
    
    return IDS, DATA, R, BOS

In [ ]:
IDS, DATA, R, BOS  = load_DANDI('../brain-lab-data/DANDI/000019', 
                               'f.acquisition["ElectricalSeries"].data[:]')

locs not available for EC9_B15
locs not available for EC9_B15
locs not available for EC9_B15
locs not available for EC9_B15
locs not available for EC9_B15
locs not available for EC9_B15
locs not available for EC9_B15
session GP31_B1 has locs
(1226807, 256)
locs not available for GP33_B1
locs not available for GP33_B1
locs not available for GP33_B1
session EC2_B1 has locs
(1974488, 256)


In [130]:
data_dir = '../brain-lab-data/DANDI/000019'
x = 'EC9'
sess = 'B53'
f = NWBHDF5IO(sorted(Path(f'{data_dir}').glob(f'sub-{x}*/*{sess}*.nwb'))[0], "r").read()
f

root pynwb.file.NWBFile at 0x140709776414272
Fields:
  acquisition: {
    ElectricalSeries <class 'pynwb.ecephys.ElectricalSeries'>
  }
  devices: {
    auto_device <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    auto_group <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  electrodes: electrodes <class 'hdmf.common.table.DynamicTable'>
  epochs: epochs <class 'pynwb.epoch.TimeIntervals'>
  file_create_date: [datetime.datetime(2019, 6, 19, 12, 29, 13, 358846, tzinfo=tzoffset(None, -25200))]
  identifier: EC9_B53
  institution: University of California, San Francisco
  intervals: {
    epochs <class 'pynwb.epoch.TimeIntervals'>,
    invalid_times <class 'pynwb.epoch.TimeIntervals'>,
    trials <class 'pynwb.epoch.TimeIntervals'>
  }
  invalid_times: invalid_times <class 'pynwb.epoch.TimeIntervals'>
  lab: Chang Lab
  session_description: EC9_B53
  session_id: EC9_B53
  session_start_time: 1900-01-01 08:00:00+00:00
  subject: subject pynwb.file.Subject at 0x140709776914432
Fields:
  species: Homo sapiens
  subject_id: EC9

  timestamps_reference_time: 1900-01-01 08:00:00+00:00
  trials: trials <class 'pynwb.epoch.TimeIntervals'>

### Kurtosis threshold

trim out the electrodes that don't pass the test. cull the brains that don't have enough electrodes

the Miller data don't include kurtosis and i think they're already trimmed

In [ ]:
# put something here when there's data that need it

### For each brain

compute fbcm excluding subject brain

### For each electrode

estimate activity and compare

multiprocessing architecture:
1. pool1 of 8 for subjects
    1. pool2 of 2 for trodes
    2. save results to file as they are completed
    3. close/join pool2
2. close/join pool1

### Make an output folder

In [103]:
try:
    out_dir = f'{data_dir}/out'
    os.mkdir(out_dir)
except:
    print('out directory exists')

out directory exists


### multi-threaded

In [115]:
BOS.keys()

dict_keys(['TWH088', 'P29HMH', 'P19HMH', 'P28HMH', 'P17HMH', 'P31CS', 'P37CS', 'P29CS', 'P39CS', 'P44HMH', 'P56CS', 'P21HMH', 'P58CS', 'P27CS', 'P9HMH', 'P47HMH', 'P51CS', 'P49CS', 'P26CS', 'P53CS', 'TWH100', 'P24CS', 'P55CS', 'P47CS', 'P62CS', 'P43HMH', 'TWH101', 'P34CS', 'P27HMH', 'P42CS', 'P14HMH', 'P33CS', 'TWH098', 'P23HMH', 'P18HMH', 'P48CS', 'P60CS', 'P54CS', 'P25CS', 'P51HMH', 'P43CS', 'P57CS', 'P48HMH', 'P38CS', 'P40CS', 'TWH107', 'P32CS', 'P42HMH', 'TWH090', 'P61CS', 'P11HMH', 'TWH103', 'P44CS'])

In [117]:
BOS['P19HMH'].get_data()

,0
0,55.0
1,1.0
2,2.0
3,3.0
4,21.0
...,...
999,2.0
1000,3.0
1001,32.0
1002,6.0


In [110]:
import pandas as pd
pd.DataFrame(DATA['TWH088']['locs'])

,0,1,2
0,31.8235,-21.7858,-15.1773
1,31.8235,-21.7858,-15.1773
2,31.8235,-21.7858,-15.1773
3,31.8235,-21.7858,-15.1773
4,31.8235,-21.7858,-15.1773
5,31.8235,-21.7858,-15.1773
6,31.8235,-21.7858,-15.1773


In [104]:
%%time
def f1(sub):
    # make a model using all the brains except the subject at all locations
    mo = se.Model([BOS[x] for x in IDS if x != sub], locs=R)
    
    # get the correlation matrix
    K = mo.get_model()

    # get subject electrode locations
    locs = DATA[sub]['locs']

    def f2(i,trode):
        # get Y_ska by removing this electrode
        Y_ska = DATA[sub]['data'][:,np.where(~np.any(locs != trode, axis=1))[0]]

        # get the indices of R where this patient's electrodes are
        aa_mask = np.array([True if np.any(np.all(x == locs, axis=1)) and (x != trode).any() else False for x in R])
        ba_mask = np.array([True if (x == trode).all() else False for x in R])

        # get K_aa and K_ba using these masks
        K_aa = K[np.ix_(aa_mask,aa_mask)]
        K_ba = K[np.ix_(ba_mask,aa_mask)]
        
        # compute Y_skb
        Y_skb = ((K_ba@np.linalg.inv(K_aa))@(Y_ska.T)).T

        # save predicted trace
        np.savetxt(f'{out_dir}/sub-{sub}_electrode-{i}_recon.csv', Y_skb)

    # iterate over the electrodes in this subject's brain
    for i,trode in enumerate(locs):
        f2(i,trode)

# iterate over the subjects
with mp.Pool(8) as p1:
    p1.map(f1, IDS)
    p1.close()
    p1.join()

IndexError: tuple index out of range

### single-threaded

In [ ]:
%%time
# iterate over the subjects
for sub in IDS:
    # make a model using all the brains except the subject at all locations
    mo = se.Model([BOS[x] for x in IDS if x != sub], locs=R)
    
    # get the correlation matrix
    K = mo.get_model()

    # get subject electrode locations
    locs = DATA[sub]['locs']

    # iterate over the electrodes in this subject's brain
    for i,trode in enumerate(locs):
        
        # get Y_ska by removing this electrode
        Y_ska = DATA[sub]['data'][:,np.unique(np.where(~(locs == trode))[0])]

        # get the indices of R where this patient's electrodes are
        aa_mask = np.array([True if np.any(np.all(x == locs, axis=1)) and (x != trode).any() else False for x in R])
        ba_mask = np.array([True if (x == trode).all() else False for x in R])
        
        # get K_aa and K_ba using these masks
        K_aa = K[np.ix_(aa_mask,aa_mask)]
        K_ba = K[np.ix_(ba_mask,aa_mask)]

        # compute Y_skb
        Y_skb = ((K_ba@np.linalg.inv(K_aa))@(Y_ska.T)).T

        # save predicted trace
        np.savetxt(f'{out_dir}/sub-{sub}_electrode-{i}_recon.csv', Y_skb)